# <p align='center'> **Footfall Cam Staff with Nametag Detection** </p> 

## **Check GPU**

In [19]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.5.1+cu121
CUDA Available: True


## **Download Dataset from Roboflow**

In [20]:
from roboflow import Roboflow
rf = Roboflow(api_key="sswqYpTPvM4EOnYQcgwm")
project = rf.workspace("lee-jia-xuan").project("footfall-cam-staff-detection")
version = project.version(4)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...


## **Check yaml**

In [21]:
import os

dataset_dir = "Footfall-Cam-Staff-Detection-4"
yaml_path = os.path.join(dataset_dir, "data.yaml")

print("data.yaml exists:", os.path.exists(yaml_path))

data.yaml exists: True


## **Fine Tune YOLOv11**

In [22]:
from ultralytics import YOLO

yaml_path = r"Footfall-Cam-Staff-Detection-4/data.yaml"

# Pretrained YOLOv11 base model
model = YOLO("yolo11n.pt")  # or "yolo11s.pt" for higher capacity

# Fine-Tuning
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=1280,                  
    batch=8,                    
    name="staff_detection_run4", 
    project="runs/detect"
)

saved_model_path = os.path.abspath(results.save_dir / "weights" / "best.pt")
print(f"\nTraining Complete! Your model path is:\n{saved_model_path}")

New https://pypi.org/project/ultralytics/8.4.124 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.120  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Footfall-Cam-Staff-Detection-4/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, m

## **YOLOv11 + SAHI Model Prediction**

In [ ]:
from pathlib import Path
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

MODEL_PATH = r"runs/detect/runs/detect/staff_detection_run4/weights/best.pt"
TEST_IMAGES_DIR = Path(r"C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\Footfall-Cam-Staff-Detection-4\test\images")
OUTPUT_DIR = "sahi_outputs"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load SAHI Model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=MODEL_PATH,
    confidence_threshold=0.25,
    device="cuda:0"  # Use "cpu" if running without GPU
)

image_paths = list(TEST_IMAGES_DIR.glob("*.jpg")) + list(TEST_IMAGES_DIR.glob("*.png")) + list(TEST_IMAGES_DIR.glob("*.jpeg"))

print(f"Found {len(image_paths)} images to process...")

# Loop through each image and run SAHI
for img_path in image_paths:
    print(f"Processing: {img_path.name}")
    
    result = get_sliced_prediction(
        image=str(img_path),  # Fixed parameter name
        detection_model=detection_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        postprocess_type="NMS",  # Switches post-processing from NMM (Greedy Merging) to standard NMS
      postprocess_match_threshold=0.4,  # IoU threshold: overlapping boxes above 40% IoU are suppressed
      postprocess_class_agnostic=False
    ) 
    
    result.export_visuals(export_dir=OUTPUT_DIR, file_name=f"sahi_{img_path.stem}")

print(f"\nFinished! Results saved to folder: '{OUTPUT_DIR}'")

Found 14 images to process...
Processing: frame_0342_jpg.rf.61591c86de50692cc743ebed44df31a5.jpg
Performing prediction on 9 slices.
Processing: frame_0426_jpg.rf.3ecc5df818217af594f30515806ec6ef.jpg
Performing prediction on 9 slices.
Processing: frame_0432_jpg.rf.a33b219f2b928a77c66f9ce60f70a192.jpg
Performing prediction on 9 slices.
Processing: frame_0507_jpg.rf.76b2b877289a615127ae2c42e4420589.jpg
Performing prediction on 9 slices.
Processing: frame_0555_jpg.rf.58a4fb20074273043d66e10933c1e622.jpg
Performing prediction on 9 slices.
Processing: frame_0936_jpg.rf.a58f359f1c9f6c007161bcedabbd476c.jpg
Performing prediction on 9 slices.
Processing: frame_1110_jpg.rf.730565788b25b68d03e21d0e86008eca.jpg
Performing prediction on 9 slices.
Processing: frame_1158_jpg.rf.0b6a01b5087a27990200b149b084147f.jpg
Performing prediction on 9 slices.
Processing: frame_1167_jpg.rf.f07e720d37113b41ae197e1bb4757fcb.jpg
Performing prediction on 9 slices.
Processing: frame_1170_jpg.rf.b7697ac122955649016592

## **YOLOv11 Prediction**

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load fine-tuned model
yolo_model = YOLO(r"runs/detect/runs/detect/staff_detection_run4/weights/best.pt")

# Directory setup
test_images_dir = Path(r"C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\Footfall-Cam-Staff-Detection-4\test\images")
output_dir = Path("yolo_outputs")
output_dir.mkdir(exist_ok=True)

# Predict without slicing
results = yolo_model.predict(
    source=str(test_images_dir),
    conf=0.25,
    imgsz=1280,
    save=True,
    project="yolo_outputs",
    name="standard_yolo",
    exist_ok=True,
    save_conf=True 
)

print("Standard YOLO results saved to 'yolo_outputs/standard_yolo'")


image 1/14 C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\Footfall-Cam-Staff-Detection-4\test\images\Screenshot-2026-08-18-221809_png.rf.884445c49ac283fe7fbb10d0eb7ae10a.jpg: 1280x1280 1 nametag, 2 persons, 12.0ms
image 2/14 C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\Footfall-Cam-Staff-Detection-4\test\images\Screenshot-2026-08-18-222459_png.rf.558bad336644e52bd0c99873872fa33e.jpg: 1280x1280 1 nametag, 2 persons, 12.3ms
image 3/14 C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\Footfall-Cam-Staff-Detection-4\test\images\Screenshot-2026-08-18-222635_png.rf.424287f84e5fc6d19695b3eed7554914.jpg: 1280x1280 1 nametag, 3 persons, 11.8ms
image 4/14 C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\Footfall-Cam-Staff-Detection-4\test\images\frame_0342_jpg.rf.61591c86de50692cc743ebed44df31a5.jpg: 1280x1280 23 persons, 11.2ms
image 5/14 C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\sr

## **Build Comparison Photo**

In [ ]:
import cv2
import numpy as np
from pathlib import Path

yolo_dir = Path(r"C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\runs\detect\yolo_outputs\standard_yolo")
sahi_dir = Path("sahi_outputs")
comparison_dir = Path("comparison_outputs")
comparison_dir.mkdir(exist_ok=True)

for yolo_img_path in yolo_dir.glob("*.jpg"):
    sahi_img_name = f"sahi_{yolo_img_path.stem}.png"
    sahi_img_path = sahi_dir / sahi_img_name
    
    if not sahi_img_path.exists():
        sahi_img_path = sahi_dir / f"sahi_{yolo_img_path.stem}.jpg"

    if sahi_img_path.exists():
        img_yolo = cv2.imread(str(yolo_img_path))
        img_sahi = cv2.imread(str(sahi_img_path))

        if img_yolo.shape != img_sahi.shape:
            img_sahi = cv2.resize(img_sahi, (img_yolo.shape[1], img_yolo.shape[0]))

        cv2.putText(img_yolo, "Standard YOLOv11 (1024x1024)", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.putText(img_sahi, "SAHI Sliced Inference (320x320)", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        combined = np.hstack((img_yolo, img_sahi))
        
        save_path = comparison_dir / f"compare_{yolo_img_path.name}"
        cv2.imwrite(str(save_path), combined)
        print(f"Saved comparison: {save_path.name}")

print("\nComparison images ready in 'comparison_outputs/'")

Saved comparison: compare_frame_0342_jpg.rf.61591c86de50692cc743ebed44df31a5.jpg
Saved comparison: compare_frame_0426_jpg.rf.3ecc5df818217af594f30515806ec6ef.jpg
Saved comparison: compare_frame_0432_jpg.rf.a33b219f2b928a77c66f9ce60f70a192.jpg
Saved comparison: compare_frame_0507_jpg.rf.76b2b877289a615127ae2c42e4420589.jpg
Saved comparison: compare_frame_0555_jpg.rf.58a4fb20074273043d66e10933c1e622.jpg
Saved comparison: compare_frame_0936_jpg.rf.a58f359f1c9f6c007161bcedabbd476c.jpg
Saved comparison: compare_frame_1110_jpg.rf.730565788b25b68d03e21d0e86008eca.jpg
Saved comparison: compare_frame_1158_jpg.rf.0b6a01b5087a27990200b149b084147f.jpg
Saved comparison: compare_frame_1167_jpg.rf.f07e720d37113b41ae197e1bb4757fcb.jpg
Saved comparison: compare_frame_1170_jpg.rf.b7697ac1229556490165923b5d2f5ab2.jpg
Saved comparison: compare_frame_1200_jpg.rf.945ab610b8fb4440dce7a1722daaa20c.jpg
Saved comparison: compare_Screenshot-2026-08-18-221809_png.rf.884445c49ac283fe7fbb10d0eb7ae10a.jpg
Saved comp

## **Interactive Comparison**

In [ ]:
import base64
from pathlib import Path


def image_to_base64(img_path):
    """Converts a local image file to base64 encoding."""
    with open(img_path, "rb") as f:
        ext = Path(img_path).suffix.lower().replace(".", "")
        mime_type = "image/png" if ext == "png" else "image/jpeg"
        return f"data:{mime_type};base64,{base64.b64encode(f.read()).decode()}"


def generate_interactive_slider(
    yolo_img_path, sahi_img_path, output_html="comparison_slider.html"
):
    img1_b64 = image_to_base64(yolo_img_path)
    img2_b64 = image_to_base64(sahi_img_path)

    html_code = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>YOLOv11 vs SAHI Comparison</title>
        <link rel="stylesheet" href="https://cdn.knightlab.com/libs/juxtapose/latest/css/juxtapose.css">
        <script src="https://cdn.knightlab.com/libs/juxtapose/latest/js/juxtapose.min.js"></script>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                background-color: #0d1117;
                color: #c9d1d9;
                display: flex;
                flex-direction: column;
                align-items: center;
                padding: 30px;
                margin: 0;
            }}
            h2 {{ margin-bottom: 20px; }}
            .wrapper {{
                width: 90%;
                max-width: 1100px;
                border: 2px solid #30363d;
                border-radius: 8px;
                overflow: hidden;
                box-shadow: 0 8px 24px rgba(0, 0, 0, 0.5);
            }}
        </style>
    </head>
    <body>
        <h2>Standard YOLOv11 (1024x1024) vs YOLOv11 + SAHI Sliced</h2>
        <div class="wrapper">
            <div id="juxtapose-container"></div>
        </div>
        <script>
            new juxtapose.JXSlider('#juxtapose-container',
                [
                    {{ src: '{img1_b64}', label: 'Standard YOLOv11' }},
                    {{ src: '{img2_b64}', label: 'YOLOv11 + SAHI Sliced' }}
                ],
                {{ animate: true, showLabels: true, showCredits: false, startingPosition: "50%" }}
            );
        </script>
    </body>
    </html>
    """

    with open(output_html, "w", encoding="utf-8") as f:
        f.write(html_code)

    print(f"Interactive comparison successfully generated: {output_html}")


generate_interactive_slider(
    yolo_img_path=r"C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\runs\detect\yolo_outputs\standard_yolo\frame_1170_jpg.rf.b7697ac1229556490165923b5d2f5ab2.jpg",
    sahi_img_path=r"C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\src\sahi_outputs\sahi_frame_1170_jpg.rf.b7697ac1229556490165923b5d2f5ab2.png",
    output_html="comparison_slider.html",
)

Interactive comparison successfully generated: comparison_slider.html


## **Video Detection**

In [ ]:
import csv
import math
from pathlib import Path
import cv2
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

MODEL_PATH = r"runs/detect/runs/detect/staff_detection_run4/weights/best.pt"
INPUT_VIDEO = (
    r"C:\Users\Jia Xuan\Documents\Portfolio\footfallcam-staff-detection\sample.mp4"
)
OUTPUT_VIDEO = "tracked_staff_clip.mp4"
CSV_OUTPUT = "staff_movement_coordinates.csv"

TARGET_WIDTH = 1280
TARGET_HEIGHT = 1280

# --- CONFIDENCE THRESHOLDS ---
PERSON_CONF_THRESH = 0.40  # Filter weak person detections
NAMETAG_CONF_THRESH = 0.50  # Strictly filter out non-nametag patterns


class StaffOnlyTracker:

    def __init__(self, max_missing_frames=5, max_distance=60):
        self.tracked_staff = {}
        self.next_id = 1
        self.max_missing_frames = max_missing_frames
        self.max_distance = max_distance

    def update(self, current_persons, detected_nametags):
        valid_persons = [
            p for p in current_persons if p["confidence"] >= PERSON_CONF_THRESH
        ]
        valid_nametags = [
            t
            for t in detected_nametags
            if t["confidence"] >= NAMETAG_CONF_THRESH
        ]

        newly_verified_staff = []

        for tag in valid_nametags:
            tcx, tcy = tag["center"]

            for person in valid_persons:
                px1, py1, px2, py2 = person["box"]
                upper_py2 = py1 + int((py2 - py1) * 0.50)  # Tightened to top 50%

                if px1 <= tcx <= px2 and py1 <= tcy <= upper_py2:
                    if person not in newly_verified_staff:
                        newly_verified_staff.append(person)
                    break

        updated_tracks = {}
        matched_person_indices = set()

        for staff_id, track_data in self.tracked_staff.items():
            tx, ty = track_data["center"]
            best_dist = float("inf")
            best_idx = -1

            for idx, p in enumerate(valid_persons):
                if idx in matched_person_indices:
                    continue
                px1, py1, px2, py2 = p["box"]
                cx, cy = int((px1 + px2) / 2.0), int((py1 + py2) / 2.0)
                dist = math.hypot(cx - tx, cy - ty)

                if dist < best_dist and dist < self.max_distance:
                    best_dist = dist
                    best_idx = idx

            if best_idx != -1:
                p = valid_persons[best_idx]
                px1, py1, px2, py2 = p["box"]
                cx, cy = int((px1 + px2) / 2.0), int((py1 + py2) / 2.0)

                updated_tracks[staff_id] = {
                    "center": (cx, cy),
                    "box": p["box"],
                    "confidence": p["confidence"],
                    "missing": 0,
                }
                matched_person_indices.add(best_idx)
            else:
                if track_data["missing"] < self.max_missing_frames:
                    track_data["missing"] += 1
                    updated_tracks[staff_id] = track_data

        for p in newly_verified_staff:
            p_box = p["box"]
            already_tracked = False

            for tid, t_data in updated_tracks.items():
                if t_data["missing"] == 0 and t_data["box"] == p_box:
                    already_tracked = True
                    break

            if not already_tracked:
                px1, py1, px2, py2 = p_box
                cx, cy = int((px1 + px2) / 2.0), int((py1 + py2) / 2.0)
                updated_tracks[self.next_id] = {
                    "center": (cx, cy),
                    "box": p_box,
                    "confidence": p["confidence"],
                    "missing": 0,
                }
                self.next_id += 1

        self.tracked_staff = updated_tracks
        return self.tracked_staff


def parse_sahi_detections(sahi_prediction):
    object_predictions = sahi_prediction.object_prediction_list
    persons = []
    nametags = []

    for pred in object_predictions:
        box = [int(v) for v in pred.bbox.to_xyxy()]
        cat_name = str(pred.category.name).lower().strip()
        conf = float(pred.score.value)

        if "person" in cat_name:
            persons.append({"box": box, "confidence": conf})
        elif any(
            tag_kw in cat_name
            for tag_kw in ["tag", "nametag", "badge", "card", "label"]
        ):
            tx1, ty1, tx2, ty2 = box
            tcx = (tx1 + tx2) / 2.0
            tcy = (ty1 + ty2) / 2.0
            nametags.append(
                {"box": box, "center": (tcx, tcy), "confidence": conf}
            )

    return persons, nametags


def draw_top_right_hud(frame, active_staff):
    h, w, _ = frame.shape
    active_items = {k: v for k, v in active_staff.items() if v["missing"] == 0}

    box_width = 320
    box_height = 40 + (max(1, len(active_items)) * 30)

    x1, y1 = w - box_width - 20, 20
    x2, y2 = w - 20, y1 + box_height

    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv2.putText(
        frame,
        "STAFF TRACKING HUD",
        (x1 + 15, y1 + 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2,
    )
    cv2.line(frame, (x1 + 10, y1 + 32), (x2 - 10, y1 + 32), (0, 255, 0), 1)

    if not active_items:
        cv2.putText(
            frame,
            "No Active Staff",
            (x1 + 15, y1 + 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (250, 250, 250),
            1,
        )
    else:
        line_y = y1 + 60
        for staff_id, data in active_items.items():
            cx, cy = data["center"]
            cv2.putText(
                frame,
                f"Staff #{staff_id} -> X: {cx}, Y: {cy}",
                (x1 + 15, line_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 255),
                2,
            )
            line_y += 30


# --- MAIN PIPELINE ---
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (TARGET_WIDTH, TARGET_HEIGHT),
)

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=MODEL_PATH,
    confidence_threshold=0.40,
    device="cuda:0",
)

tracker = StaffOnlyTracker(max_missing_frames=5, max_distance=60)
frame_count = 0

with open(CSV_OUTPUT, mode="w", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(
        [
            "Frame",
            "Timestamp_Sec",
            "Staff_ID",
            "X_Center",
            "Y_Center",
            "Confidence",
        ]
    )

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        timestamp = round(frame_count / fps, 2)

        resized_frame = cv2.resize(
            frame, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_CUBIC
        )

        sahi_result = get_sliced_prediction(
            image=resized_frame,
            detection_model=detection_model,
            slice_height=640,
            slice_width=640,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
            postprocess_type="NMS",
            postprocess_match_threshold=0.4,
            postprocess_class_agnostic=False,
        )

        persons, nametags = parse_sahi_detections(sahi_result)
        active_staff = tracker.update(persons, nametags)

        render_hud_dict = {}
        for staff_id, data in active_staff.items():
            if data["missing"] == 0:
                x1, y1, x2, y2 = data["box"]
                cx, cy = data["center"]
                conf = data["confidence"]

                csv_writer.writerow(
                    [frame_count, timestamp, staff_id, cx, cy, f"{conf:.2f}"]
                )

                cv2.rectangle(
                    resized_frame, (x1, y1), (x2, y2), (0, 255, 0), 2
                )
                cv2.circle(resized_frame, (cx, cy), 6, (0, 0, 255), -1)
                cv2.putText(
                    resized_frame,
                    f"Staff #{staff_id}",
                    (x1, max(y1 - 10, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2,
                )
                render_hud_dict[staff_id] = data

        draw_top_right_hud(resized_frame, render_hud_dict)
        out.write(resized_frame)

        print(
            f"Frame {frame_count}: Persons={len(persons)}, Tags={len(nametags)} | Tracked Staff={len(render_hud_dict)}"
        )

cap.release()
out.release()
print("Processing complete.")

Performing prediction on 9 slices.
Frame 1: Persons=23, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 2: Persons=27, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 3: Persons=24, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 4: Persons=25, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 5: Persons=22, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 6: Persons=20, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 7: Persons=20, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 8: Persons=16, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 9: Persons=20, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 10: Persons=18, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 11: Persons=19, Tags=0 | Tracked Staff=0
Performing prediction on 9 slices.
Frame 12: Persons=21, Tags=0 | Tracked Staff=0
Performing prediction on 